In [1]:
!pip install -q -U "transformers>=4.49.0" accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.2 MB/s eta 0:00:00:00:01


In [2]:

# Cell 1

import numpy as np
import pandas as pd
import torch
import joblib
import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [3]:
import pandas as pd

data_path = "/kaggle/input/datasets/joelleiliovits/new-data-csv/new_data.csv"
df = pd.read_csv(data_path)

In [4]:
df = df[["text", "tags"]].copy()
df = df.dropna(subset=["text", "tags"])

df["text"] = df["text"].astype(str).str.strip()
df["tags"] = df["tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["tags"] != "")]



In [5]:
df["label_list"] = df["tags"].apply(lambda x: x.split())
df[["text", "tags", "label_list"]].head()  


,text,tags,label_list
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]"
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]"
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]"
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]"
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]"


In [6]:
df["num_labels"] = df["label_list"].apply(len)

print(df["num_labels"].value_counts().sort_index())
df[["text", "tags", "label_list", "num_labels"]].head()

num_labels
1    1000000
2     701000
3     112707
4       7949
5        304
Name: count, dtype: int64


,text,tags,label_list,num_labels
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]",2
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]",2
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]",2
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]",2
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]",2


In [7]:
df_1 = df[df["num_labels"] == 1].copy()
df_2 = df[df["num_labels"] == 2].copy()
df_3_plus = df[df["num_labels"] >= 3].copy()



In [8]:
RANDOM_STATE = 42

df_1_sample = df_1.sample(
    n=min(800000, len(df_1)),
    random_state=RANDOM_STATE
).copy()

df_2_sample = df_2.sample(
    n=min(320000, len(df_2)),
    random_state=RANDOM_STATE
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final shape:", df_final.shape)
print(df_final["num_labels"].value_counts().sort_index())

Final shape: (1240960, 4)
num_labels
1    800000
2    320000
3    112707
4      7949
5       304
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 52
y shape: (1240960, 52)
First labels: ['apache' 'asp.net-core' 'authentication' 'azure' 'bash' 'c#'
 'computer-vision' 'cors' 'cuda' 'debugging' 'deep-learning' 'django'
 'dns' 'docker' 'fastapi' 'firewall' 'gpu' 'http' 'inference' 'java']


In [10]:
X = df_final["text"].tolist()



In [11]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)



In [12]:
from datasets import Dataset
import numpy as np

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_dataset = Dataset.from_dict({
    "text": X_train if isinstance(X_train, list) else X_train.tolist(),
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val if isinstance(X_val, list) else X_val.tolist(),
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test if isinstance(X_test, list) else X_test.tolist(),
    "labels": y_test.tolist()
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 992768
Val: 124096
Test: 124096


In [13]:
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [14]:
# df فيه عمود text = title + body
sample_df = df.sample(n=min(100_000, len(df)), random_state=42).copy()

texts = sample_df["text"].fillna("").astype(str).tolist()

lengths = []
batch_size = 512

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    enc = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    lengths.extend(len(x) for x in enc["input_ids"])

lengths = np.array(lengths)

print("count:", len(lengths))
print("min:", lengths.min())
print("median:", int(np.percentile(lengths, 50)))
print("p90:", int(np.percentile(lengths, 90)))
print("p95:", int(np.percentile(lengths, 95)))
print("p99:", int(np.percentile(lengths, 99)))
print("max:", lengths.max())

print("over_64 :", round((lengths > 64).mean() * 100, 2), "%")
print("over_128:", round((lengths > 128).mean() * 100, 2), "%")
print("over_256:", round((lengths > 256).mean() * 100, 2), "%")

count: 100000
min: 13
median: 187
p90: 450
p95: 560
p99: 783
max: 1662
over_64 : 94.55 %
over_128: 71.1 %
over_256: 32.63 %


In [15]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=150
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/992768 [00:00<?, ? examples/s]

Map:   0%|          | 0/124096 [00:00<?, ? examples/s]

Map:   0%|          | 0/124096 [00:00<?, ? examples/s]

In [16]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 52


In [17]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/unixcode_results",
    eval_strategy="steps",
    eval_steps=5000,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
    save_only_model=False,

    per_device_train_batch_size=44,
    per_device_eval_batch_size=44,

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,

    fp16=True,
    bf16=False,

    report_to="none"
)

In [18]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLD = 0.35

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    probs = sigmoid(logits)
    preds_thr = (probs >= THRESHOLD).astype(int)

    results["precision_threshold"] = precision_score(labels, preds_thr, average="micro", zero_division=0)
    results["recall_threshold"] = recall_score(labels, preds_thr, average="micro", zero_division=0)
    results["f1_threshold"] = f1_score(labels, preds_thr, average="micro", zero_division=0)

    for k in [1, 2, 3, 4, 5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
5000,0.045922,0.045797,0.833537,0.866942,0.849911,0.921021,0.630145,0.748310,0.633884,0.867383,0.732475,0.458280,0.940638,0.616298,0.352914,0.965828,0.516939,0.285550,0.976839,0.441918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)